In [94]:
!pip install numpy torch torchsummary scikit-learn pandas plotly

In [95]:
# Importar bibliotecas necessárias

# Deep Learning / Machine Learning

import numpy as np
import torch
import torch.nn as nn
from torchsummary import summary
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# EDA
import pandas as pd
import plotly.express as px

## Carregar os dados

In [96]:
# Carregar o dataset
df_veiculos = pd.read_csv('data/veiculos.csv')

## Exploração Inicial dos dados

In [97]:
# Estrutura dos dados
df_veiculos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 835 entries, 0 to 834
Data columns (total 17 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Categoria                        835 non-null    int64  
 1   Cor                              835 non-null    object 
 2   Pais de Origem                   835 non-null    object 
 3   Ano Modelo                       835 non-null    int64  
 4   Ano Fabricação                   835 non-null    int64  
 5   Potencia                         835 non-null    int64  
 6   Quantidade de lugares            835 non-null    int64  
 7   Unico dono?                      835 non-null    int64  
 8   Ja teve sinistro?                835 non-null    int64  
 9   Ja foi carro de aplicativo?      835 non-null    int64  
 10  Revisoes em dia?                 835 non-null    int64  
 11  Sistema avancado de Multimidia?  835 non-null    int64  
 12  Tipo de Motorizacao   

In [98]:
# Visualizar primeiras linhas
df_veiculos.head()

,Categoria,Cor,Pais de Origem,Ano Modelo,Ano Fabricação,Potencia,Quantidade de lugares,Unico dono?,Ja teve sinistro?,Ja foi carro de aplicativo?,Revisoes em dia?,Sistema avancado de Multimidia?,Tipo de Motorizacao,Kilometragem,Tipo de Transmissao,Tamanho do porta malas,Valor de Venda
0,4,azul,Alemanha,2026,2025,143,4,0,0,1,1,1,Híbrido,60459,3,430,112898.39
1,7,verde,Alemanha,2024,2024,541,5,0,0,1,1,0,Híbrido,105982,7,484,887822.26
2,2,prata,Japão,2026,2025,94,4,0,1,1,0,0,Flex,38626,3,321,55516.43
3,4,preto,Japão,2022,2021,159,4,1,1,0,0,1,Flex,91185,2,415,147030.87
4,2,azul,Coreia do Sul,2026,2025,114,4,0,0,0,0,1,Flex,26037,3,381,93719.67


In [99]:
# Estatísticas
df_veiculos.describe()

,Categoria,Ano Modelo,Ano Fabricação,Potencia,Quantidade de lugares,Unico dono?,Ja teve sinistro?,Ja foi carro de aplicativo?,Revisoes em dia?,Sistema avancado de Multimidia?,Kilometragem,Tipo de Transmissao,Tamanho do porta malas,Valor de Venda
count,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,8.350000e+02
mean,4.453892,2024.037126,2023.428743,307.476647,4.426347,0.492216,0.534132,0.462275,0.489820,0.489820,96439.767665,3.785629,395.049102,7.902733e+05
std,2.274606,1.402229,1.358211,320.806721,1.303114,0.500239,0.499133,0.498874,0.500196,0.500196,78575.237811,2.169836,131.769333,1.742400e+06
min,1.000000,2022.000000,2021.000000,70.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,12611.000000,1.000000,80.000000,4.280000e+04
25%,2.000000,2023.000000,2022.000000,114.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,42289.500000,2.000000,308.000000,8.187166e+04
50%,4.000000,2024.000000,2024.000000,160.000000,4.000000,0.000000,1.000000,0.000000,0.000000,0.000000,70509.000000,3.000000,428.000000,1.436343e+05
75%,6.000000,2025.000000,2025.000000,371.500000,5.000000,1.000000,1.000000,1.000000,1.000000,1.000000,129393.500000,5.000000,490.500000,4.210006e+05
max,8.000000,2026.000000,2025.000000,1485.000000,7.000000,1.000000,1.000000,1.000000,1.000000,1.000000,403947.000000,9.000000,600.000000,9.433305e+06


In [100]:
# Mostrar os valores únicos das variáveis categoricas
for column in df_veiculos.select_dtypes(include=['object']).columns:
    print(f'{column}: {df_veiculos[column].unique()}')

Cor: ['azul' 'verde' 'prata' 'preto' 'cinza' 'vermelho' 'branco']
Pais de Origem: ['Alemanha' 'Japão' 'Coreia do Sul' 'Inglaterra' 'França' 'Itália'
 'Estados Unidos' 'China']
Tipo de Motorizacao: ['Híbrido' 'Flex' 'Elétrico' 'Gasolina']


In [101]:
"""Domínios
Categoria
1 - Econômico / Compacto
2 - Intermediário / Hatch Médio
3 - Sedan Compacto
4 - SUV de Entrada
5 - Sedan / SUV Médio
6 - Premium / Executivo
7 - Luxo / Superluxo
8 - Superesportivo / Hipercarro
Tipo de Transmissão
1 - Manual
2 - Automático
3 - CVT
4 - Automático 8 marchas
5 - Automático 10 marchas
6 - DCT
7 - Eletrônica
8 - Automático Esportivo
9 - Sequencial Paddle Shift
"""

'Domínios\nCategoria\n1 - Econômico / Compacto\n2 - Intermediário / Hatch Médio\n3 - Sedan Compacto\n4 - SUV de Entrada\n5 - Sedan / SUV Médio\n6 - Premium / Executivo\n7 - Luxo / Superluxo\n8 - Superesportivo / Hipercarro\nTipo de Transmissão\n1 - Manual\n2 - Automático\n3 - CVT\n4 - Automático 8 marchas\n5 - Automático 10 marchas\n6 - DCT\n7 - Eletrônica\n8 - Automático Esportivo\n9 - Sequencial Paddle Shift\n'

## Preparação de Dados para EDA

In [102]:
# Criar lista de varíaveis categóricas
categorical_features = df_veiculos.select_dtypes(include=['object']).columns.tolist()

# Incluir variáveis categoricas adicionais
additional_categorical_features = ['Categoria', 'Ano Modelo', 'Ano Fabricação', 'Unico dono?', 'Ja teve sinistro?', 'Ja foi carro de aplicativo?', 'Revisoes em dia?', 'Sistema avancado de Multimidia?', 'Tipo de Transmissao']
categorical_features.extend(additional_categorical_features)
categorical_features

['Cor',
 'Pais de Origem',
 'Tipo de Motorizacao',
 'Categoria',
 'Ano Modelo',
 'Ano Fabricação',
 'Unico dono?',
 'Ja teve sinistro?',
 'Ja foi carro de aplicativo?',
 'Revisoes em dia?',
 'Sistema avancado de Multimidia?',
 'Tipo de Transmissao']

In [103]:
# Criar uma lista de variáveis numéricas
numerical_features = df_veiculos.select_dtypes(include=['int64', 'float64']).columns.tolist()
# Remover o Target
numerical_features.remove('Valor de Venda')
# Remover features categoricas do tipo numérico
numerical_features = [feature for feature in numerical_features if feature not in categorical_features]
numerical_features

['Potencia', 'Quantidade de lugares', 'Kilometragem', 'Tamanho do porta malas']

In [104]:
# Variável target
target = ['Valor de Venda']

## EDA

In [105]:
# Distribuição da variável, target, usando plotly
fig = px.histogram(df_veiculos, x=target, nbins=50, title='Distribuição do Valor de Venda')
fig.show()

In [106]:
# Distribuição das variáveis numéricas
for feature in numerical_features:
    fig = px.histogram(df_veiculos, x=feature, nbins=50, title=f'Distribuição de {feature}')
    fig.show()

In [107]:
# BoxPlot das variáveis numéricas
for feature in numerical_features:
    fig = px.box(df_veiculos, y=feature, title=f'BoxPlot de {feature}')
    fig.show()

In [108]:
# Distribuição das variáveis categoricas
for feature in categorical_features:
    fig = px.histogram(df_veiculos, x=feature, title=f'Distribuição de {feature}')
    fig.show()

In [109]:
# Boxplot das variáveis categoricas com o target
for feature in categorical_features:
    fig = px.box(df_veiculos, x=feature, y=target, title=f'BoxPlot de {feature} com {target}')
    fig.show()

## Preparar dados para Correlações

In [110]:
# Transformar variáveis categóricas originais do dataset, usando one-hot encoding
df_veiculos_encoded = pd.get_dummies(df_veiculos, dtype=int)
df_veiculos_encoded.head()

,Categoria,Ano Modelo,Ano Fabricação,Potencia,Quantidade de lugares,Unico dono?,Ja teve sinistro?,Ja foi carro de aplicativo?,Revisoes em dia?,Sistema avancado de Multimidia?,...,Pais de Origem_Coreia do Sul,Pais de Origem_Estados Unidos,Pais de Origem_França,Pais de Origem_Inglaterra,Pais de Origem_Itália,Pais de Origem_Japão,Tipo de Motorizacao_Elétrico,Tipo de Motorizacao_Flex,Tipo de Motorizacao_Gasolina,Tipo de Motorizacao_Híbrido
0,4,2026,2025,143,4,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,1
1,7,2024,2024,541,5,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,1
2,2,2026,2025,94,4,0,1,1,0,0,...,0,0,0,0,0,1,0,1,0,0
3,4,2022,2021,159,4,1,1,0,0,1,...,0,0,0,0,0,1,0,1,0,0
4,2,2026,2025,114,4,0,0,0,0,1,...,1,0,0,0,0,0,0,1,0,0


In [111]:
# Estrutura do dataset
df_veiculos_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 835 entries, 0 to 834
Data columns (total 33 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Categoria                        835 non-null    int64  
 1   Ano Modelo                       835 non-null    int64  
 2   Ano Fabricação                   835 non-null    int64  
 3   Potencia                         835 non-null    int64  
 4   Quantidade de lugares            835 non-null    int64  
 5   Unico dono?                      835 non-null    int64  
 6   Ja teve sinistro?                835 non-null    int64  
 7   Ja foi carro de aplicativo?      835 non-null    int64  
 8   Revisoes em dia?                 835 non-null    int64  
 9   Sistema avancado de Multimidia?  835 non-null    int64  
 10  Kilometragem                     835 non-null    int64  
 11  Tipo de Transmissao              835 non-null    int64  
 12  Tamanho do porta malas

In [112]:
# Mostrar correlação entre as variáveis, usando plotly
fig = px.imshow(df_veiculos_encoded.corr(), text_auto=True, aspect="auto",title='Correlação entre as variáveis', width=1080, height=900)
fig.show()

In [113]:
# Transformar variáveis categóricas da lista do dataset, usando one-hot encoding
df_veiculos_encoded = pd.get_dummies(df_veiculos, columns=categorical_features, dtype=int)
df_veiculos_encoded.head()

,Potencia,Quantidade de lugares,Kilometragem,Tamanho do porta malas,Valor de Venda,Cor_azul,Cor_branco,Cor_cinza,Cor_prata,Cor_preto,...,Sistema avancado de Multimidia?_1,Tipo de Transmissao_1,Tipo de Transmissao_2,Tipo de Transmissao_3,Tipo de Transmissao_4,Tipo de Transmissao_5,Tipo de Transmissao_6,Tipo de Transmissao_7,Tipo de Transmissao_8,Tipo de Transmissao_9
0,143,4,60459,430,112898.39,1,0,0,0,0,...,1,0,0,1,0,0,0,0,0,0
1,541,5,105982,484,887822.26,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,94,4,38626,321,55516.43,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
3,159,4,91185,415,147030.87,0,0,0,0,1,...,1,0,1,0,0,0,0,0,0,0
4,114,4,26037,381,93719.67,1,0,0,0,0,...,1,0,0,1,0,0,0,0,0,0


In [114]:
df_veiculos_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 835 entries, 0 to 834
Data columns (total 61 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Potencia                           835 non-null    int64  
 1   Quantidade de lugares              835 non-null    int64  
 2   Kilometragem                       835 non-null    int64  
 3   Tamanho do porta malas             835 non-null    int64  
 4   Valor de Venda                     835 non-null    float64
 5   Cor_azul                           835 non-null    int64  
 6   Cor_branco                         835 non-null    int64  
 7   Cor_cinza                          835 non-null    int64  
 8   Cor_prata                          835 non-null    int64  
 9   Cor_preto                          835 non-null    int64  
 10  Cor_verde                          835 non-null    int64  
 11  Cor_vermelho                       835 non-null    int64  

In [115]:
# Mostrar correlação entre as variáveis, usando plotly
fig = px.imshow(df_veiculos_encoded.corr(), text_auto=True, aspect="auto",title='Correlação entre as variáveis', width=1080, height=900)
fig.show()

## Preparar dados para treinamento da rede neural

In [116]:
# Transformar variáveis categóricas originais do dataset, usando one-hot encoding
df_veiculos_encoded = pd.get_dummies(df_veiculos, dtype=int)
df_veiculos_encoded.head()

,Categoria,Ano Modelo,Ano Fabricação,Potencia,Quantidade de lugares,Unico dono?,Ja teve sinistro?,Ja foi carro de aplicativo?,Revisoes em dia?,Sistema avancado de Multimidia?,...,Pais de Origem_Coreia do Sul,Pais de Origem_Estados Unidos,Pais de Origem_França,Pais de Origem_Inglaterra,Pais de Origem_Itália,Pais de Origem_Japão,Tipo de Motorizacao_Elétrico,Tipo de Motorizacao_Flex,Tipo de Motorizacao_Gasolina,Tipo de Motorizacao_Híbrido
0,4,2026,2025,143,4,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,1
1,7,2024,2024,541,5,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,1
2,2,2026,2025,94,4,0,1,1,0,0,...,0,0,0,0,0,1,0,1,0,0
3,4,2022,2021,159,4,1,1,0,0,1,...,0,0,0,0,0,1,0,1,0,0
4,2,2026,2025,114,4,0,0,0,0,1,...,1,0,0,0,0,0,0,1,0,0


In [117]:
# Dividir o dataset entre X e y
X = df_veiculos.drop(columns=target, axis=1)
y = np.array(df_veiculos_encoded[target])

In [118]:
X

,Categoria,Cor,Pais de Origem,Ano Modelo,Ano Fabricação,Potencia,Quantidade de lugares,Unico dono?,Ja teve sinistro?,Ja foi carro de aplicativo?,Revisoes em dia?,Sistema avancado de Multimidia?,Tipo de Motorizacao,Kilometragem,Tipo de Transmissao,Tamanho do porta malas
0,4,azul,Alemanha,2026,2025,143,4,0,0,1,1,1,Híbrido,60459,3,430
1,7,verde,Alemanha,2024,2024,541,5,0,0,1,1,0,Híbrido,105982,7,484
2,2,prata,Japão,2026,2025,94,4,0,1,1,0,0,Flex,38626,3,321
3,4,preto,Japão,2022,2021,159,4,1,1,0,0,1,Flex,91185,2,415
4,2,azul,Coreia do Sul,2026,2025,114,4,0,0,0,0,1,Flex,26037,3,381
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
830,5,prata,Coreia do Sul,2026,2025,244,5,1,0,1,0,0,Híbrido,43509,2,522
831,6,prata,Japão,2022,2021,258,5,0,0,1,0,1,Flex,210838,6,464
832,5,vermelho,Coreia do Sul,2023,2022,201,5,1,1,1,1,1,Híbrido,272508,3,471
833,5,verde,Estados Unidos,2024,2023,241,5,0,0,0,1,0,Elétrico,45229,4,471


In [119]:
# Dividir entre treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42, shuffle=True)

In [120]:
# Dividir entre validação e teste
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42, shuffle=True)

In [121]:
# Criar Pipeline de pré-processamento dos dados
reduced_categorical_features = ['Categoria', 'Cor', 'Ano Modelo', 'Ano Fabricação', 'Tipo de Transmissao', 'Pais de Origem', 'Tipo de Motorizacao']

# Aplicar transformação por tipo
numeric_transformer = MinMaxScaler()
categorical_transformer = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Aplicar transformação por coluna
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, reduced_categorical_features)
    ]
)

In [122]:
# Aplicar preprocessor nos splits
X_train = preprocessor.fit_transform(X_train)
X_val = preprocessor.transform(X_val)
X_test = preprocessor.transform(X_test)

scaler_y = MinMaxScaler()
y_train = scaler_y.fit_transform(y_train.reshape(-1, 1))
y_val = scaler_y.transform(y_val.reshape(-1, 1))
y_test = scaler_y.transform(y_test.reshape(-1, 1))

In [123]:
# Shape dos datasets
print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print(f'X_val: {X_val.shape}')
print(f'y_val: {y_val.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_test: {y_test.shape}')

X_train: (417, 50)
y_train: (417, 1)
X_val: (209, 50)
y_val: (209, 1)
X_test: (209, 50)
y_test: (209, 1)


In [124]:
# Salvar Preporcessor para uso futuro
import joblib
joblib.dump(preprocessor, 'preprocessor.pkl')

['preprocessor.pkl']

In [125]:
# Criar uma estrutura do dataset de Veículos em uma classe Dataset do Pytorch
class VeiculosDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [126]:
# Criar Datasets
dataset_train = VeiculosDataset(X_train, y_train)
dataset_val = VeiculosDataset(X_val, y_val)
dataset_test = VeiculosDataset(X_test, y_test)

In [127]:
# Criar Dataloaders
dataloader_train = DataLoader(dataset_train, batch_size=32, drop_last=True)
dataloader_val = DataLoader(dataset_val, batch_size=32, drop_last=True)
dataloader_test = DataLoader(dataset_test, batch_size=32, drop_last=True)

## Definir arquitetura da rede

In [133]:
# Criar uma arquitetura de rede neural com Pytorch
class NeuralNetwork(nn.Module):
    def __init__(self, input_size, hidden_layer_sizes=[128, 64, 32, 16], output_size=1, dropout_rate=0.2):
      super(NeuralNetwork, self).__init__()
      self.layer1 = nn.Linear(input_size, hidden_layer_sizes[0])
      self.bn1 = nn.BatchNorm1d(hidden_layer_sizes[0])
      self.dropout1 = nn.Dropout(dropout_rate)

      self.layer2 = nn.Linear(hidden_layer_sizes[0], hidden_layer_sizes[1])
      self.bn2 = nn.BatchNorm1d(hidden_layer_sizes[1])
      self.dropout2 = nn.Dropout(dropout_rate)

      self.layer3 = nn.Linear(hidden_layer_sizes[1], hidden_layer_sizes[2])
      self.bn3 = nn.BatchNorm1d(hidden_layer_sizes[2])
      self.dropout3 = nn.Dropout(dropout_rate)

      self.layer4 = nn.Linear(hidden_layer_sizes[2], hidden_layer_sizes[3])
      self.bn4 = nn.BatchNorm1d(hidden_layer_sizes[3])
      self.dropout4 = nn.Dropout(dropout_rate)

      self.output_layer = nn.Linear(hidden_layer_sizes[3], output_size)
      self.relu = nn.ReLU()

    def forward(self, x):
      x = self.relu(self.bn1(self.layer1(x)))
      x = self.dropout1(x)

      x = self.relu(self.bn2(self.layer2(x)))
      x = self.dropout2(x)

      x = self.relu(self.bn3(self.layer3(x)))
      x = self.dropout3(x)

      x = self.relu(self.bn4(self.layer4(x)))
      x = self.dropout4(x)

      x = self.output_layer(x)
      return x

In [134]:
# Instanciar o modelo
model = NeuralNetwork(input_size=X_train.shape[1], hidden_layer_sizes=[64, 32, 16, 8], output_size=1)

In [135]:
# Visualizar a arquitetura do modelo
summary(model, (X_train.shape[1],))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 64]           3,264
       BatchNorm1d-2                   [-1, 64]             128
              ReLU-3                   [-1, 64]               0
           Dropout-4                   [-1, 64]               0
            Linear-5                   [-1, 32]           2,080
       BatchNorm1d-6                   [-1, 32]              64
              ReLU-7                   [-1, 32]               0
           Dropout-8                   [-1, 32]               0
            Linear-9                   [-1, 16]             528
      BatchNorm1d-10                   [-1, 16]              32
             ReLU-11                   [-1, 16]               0
          Dropout-12                   [-1, 16]               0
           Linear-13                    [-1, 8]             136
      BatchNorm1d-14                   

## Treinar a Rede Neural

In [136]:
# treinar a rede
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

NUM_EPOCHS = 1000
train_losses = []
val_losses = []

for epoch in range(NUM_EPOCHS):
  model.train()
  running_train_loss = 0.0
  for data in dataloader_train:
    # Zerar os gradientes
    optimizer.zero_grad()

    # Dividir entre input e output
    inputs, targets = data

    # Forward pass
    outputs = model(inputs)

    # Calcular a perda
    loss = criterion(outputs, targets)

    # Backward pass
    loss.backward()

    # Atualizar os pesos
    optimizer.step()

    running_train_loss += loss.item()

  epoch_train_loss = running_train_loss / len(dataloader_train)
  train_losses.append(epoch_train_loss)

  # Fase de validação
  model.eval()
  running_val_loss = 0.0
  with torch.no_grad():
    for data in dataloader_val:
      # Dividir entre input e output
      inputs, targets = data

      # Forward Pass
      outputs = model(inputs)

      # Calcular a perda
      loss = criterion(outputs, targets)
      running_val_loss += loss.item()

  epoch_val_loss = running_val_loss / len(dataloader_val)
  val_losses.append(epoch_val_loss)
  if epoch % 10 == 0:
    print(f'Epoch {epoch}, Train Loss: {epoch_train_loss:.6f}, Val Loss: {epoch_val_loss:.6f}')




Epoch 0, Train Loss: 0.300204, Val Loss: 0.096173
Epoch 10, Train Loss: 0.059093, Val Loss: 0.026923
Epoch 20, Train Loss: 0.034349, Val Loss: 0.019810
Epoch 30, Train Loss: 0.028228, Val Loss: 0.015862
Epoch 40, Train Loss: 0.024248, Val Loss: 0.013410
Epoch 50, Train Loss: 0.020918, Val Loss: 0.012381
Epoch 60, Train Loss: 0.023648, Val Loss: 0.010737
Epoch 70, Train Loss: 0.017861, Val Loss: 0.010323
Epoch 80, Train Loss: 0.015061, Val Loss: 0.009863
Epoch 90, Train Loss: 0.017406, Val Loss: 0.010025
Epoch 100, Train Loss: 0.016851, Val Loss: 0.009488
Epoch 110, Train Loss: 0.016691, Val Loss: 0.010020
Epoch 120, Train Loss: 0.019503, Val Loss: 0.009411
Epoch 130, Train Loss: 0.019192, Val Loss: 0.009888
Epoch 140, Train Loss: 0.016712, Val Loss: 0.008977
Epoch 150, Train Loss: 0.014652, Val Loss: 0.009072
Epoch 160, Train Loss: 0.017947, Val Loss: 0.009250
Epoch 170, Train Loss: 0.017791, Val Loss: 0.008911
Epoch 180, Train Loss: 0.015782, Val Loss: 0.008898
Epoch 190, Train Loss: 

## Visualizar resultados do treinamento

In [137]:
# Plotar loss de treino e validação com Plotly
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(1, NUM_EPOCHS+1)), y=train_losses, mode='lines', name='Train Loss'))
fig.add_trace(go.Scatter(x=list(range(1, NUM_EPOCHS+1)), y=val_losses, mode='lines', name='Val Loss'))
fig.update_layout(title='Loss de Treino e Validação', xaxis_title='Epoch', yaxis_title='Loss')
fig.show()